# 🏦 Pandas for auditors — Solutions Level 3: Medium+

**Context**: Compliance / AML/CFT — enrichment, alert detection, scoring.

> ⚠️ This file contains the **solutions**. Try first with `exercice_moyen_plus.ipynb`!

## 0. Data generation

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
np.random.seed(303)

n_clients = 60
client_ids = [f'CLI{str(i).zfill(4)}' for i in range(1, n_clients + 1)]

clients = pd.DataFrame({
    'client_id':       client_ids,
    'segment':         np.random.choice(['Individual', 'Corporate', 'Private Banking'],
                                        n_clients, p=[0.45, 0.35, 0.20]),
    'residence_country':  np.random.choice(['FR', 'LU', 'CH', 'MC', 'BE', 'DE'],
                                        n_clients, p=[0.50, 0.15, 0.12, 0.08, 0.10, 0.05]),
    'kyc_risk_level': np.random.choice(['Low', 'Standard', 'High'],
                                          n_clients, p=[0.40, 0.45, 0.15]),
    'relationship_start_date': pd.to_datetime('2015-01-01') + pd.to_timedelta(
        np.random.randint(0, 3285, n_clients), unit='D'
    ),
})

n = 600
counterparty_country = np.random.choice(
    ['FR', 'DE', 'BE', 'LU', 'CH', 'US', 'GB', 'AE', 'PA', 'CY', 'MT', 'SG'],
    n, p=[0.20, 0.12, 0.10, 0.10, 0.08, 0.08, 0.07, 0.06, 0.05, 0.05, 0.05, 0.04]
)
op_types = np.random.choice(
    ['Incoming Transfer', 'Outgoing Transfer', 'Cash Deposit', 'Cash Withdrawal',
     'Check', 'Direct Debit'],
    n, p=[0.28, 0.28, 0.10, 0.10, 0.12, 0.12]
)
dates = pd.to_datetime('2024-01-01') + pd.to_timedelta(
    np.random.randint(0, 366, n), unit='D'
)
base_amounts = np.round(np.random.lognormal(mean=7.0, sigma=1.3, size=n), 2)

flows = pd.DataFrame({
    'flow_id':          [f'FL{str(i).zfill(6)}' for i in range(1, n + 1)],
    'client_id':        np.random.choice(client_ids, n),
    'date':             dates,
    'operation_type':   op_types,
    'amount':           base_amounts,
    'currency':         np.random.choice(['EUR', 'USD', 'CHF', 'GBP'],
                                         n, p=[0.78, 0.10, 0.08, 0.04]),
    'counterparty_country': counterparty_country,
    'channel':          np.random.choice(['SWIFT', 'SEPA', 'Internal', 'Counter'],
                                         n, p=[0.25, 0.40, 0.20, 0.15]),
})

idx_struct = np.random.choice(flows.index, 12, replace=False)
flows.loc[idx_struct, 'amount'] = np.random.choice([9500, 9750, 9800, 9900, 9950, 9990], 12)
flows.loc[idx_struct, 'operation_type'] = 'Cash Deposit'

idx_cash = np.random.choice(flows.index, 8, replace=False)
flows.loc[idx_cash, 'amount'] = np.random.choice([15000, 20000, 25000, 30000], 8)
flows.loc[idx_cash, 'operation_type'] = np.random.choice(['Cash Deposit', 'Cash Withdrawal'], 8)

risk_country_list = ['AE', 'PA', 'CY']
idx_risk = np.random.choice(flows.index, 15, replace=False)
flows.loc[idx_risk, 'counterparty_country'] = np.random.choice(risk_country_list, 15)
flows.loc[idx_risk, 'amount'] = np.round(np.random.lognormal(mean=9.0, sigma=0.8, size=15), 2)

flows.loc[np.random.choice(flows.index, 15, replace=False), 'counterparty_country'] = np.nan
flows = flows.sample(frac=1, random_state=5).reset_index(drop=True)

print('Flows ready:', flows.shape)
print('Clients ready:', clients.shape)

---
## Exercise 1 — Enrichment with `merge`

In [ ]:
# 1. Merge flows + clients
flows_enriched = flows.merge(clients, on='client_id', how='left')
print('Columns after merge:', flows_enriched.shape[1])
flows_enriched.head(3)

In [ ]:
# 2. Flows with client not found (NaN in kyc_risk_level after merge)
orphans = flows_enriched[flows_enriched['kyc_risk_level'].isna()]
print('Flows without a client in the reference table:', len(orphans))

In [ ]:
# 3. Total amount by KYC risk level
flows_enriched.groupby('kyc_risk_level')['amount'].sum().round(2).sort_values(ascending=False)

In [ ]:
# 4. Private Banking flows outside France
pb_outside_fr = flows_enriched[
    (flows_enriched['segment'] == 'Private Banking')
    & (flows_enriched['residence_country'] != 'FR')
]
print('Private Banking flows outside France:', len(pb_outside_fr))
pb_outside_fr[['flow_id', 'client_id', 'residence_country', 'operation_type', 'amount']].head()

---
## Exercise 2 — Time-based analysis

In [ ]:
# 1. Extract month, weekday, day_name
flows_enriched['month']        = flows_enriched['date'].dt.month
flows_enriched['weekday']      = flows_enriched['date'].dt.dayofweek   # 0=Monday … 6=Sunday
flows_enriched['day_name']     = flows_enriched['date'].dt.day_name()
flows_enriched[['date', 'month', 'weekday', 'day_name']].head()

In [ ]:
# 2. Total amount per month
by_month = flows_enriched.groupby('month')['amount'].sum().round(2)
print('Most active month:', by_month.idxmax(), '— total:', by_month.max())
by_month

In [ ]:
# 3. Weekend flows
weekend = flows_enriched[flows_enriched['weekday'] >= 5]
pct = len(weekend) / len(flows_enriched) * 100
print(f'Weekend flows: {len(weekend)} ({pct:.1f}% of total)')

In [ ]:
# 4. Cash on weekends
weekend_cash = weekend[
    weekend['operation_type'].isin(['Cash Deposit', 'Cash Withdrawal'])
]
print('Cash operations on weekends:', len(weekend_cash))
weekend_cash[['date', 'day_name', 'client_id', 'operation_type', 'amount']]

---
## Exercise 3 — Detecting *structuring*

In [ ]:
# 1. Flows in the 9,000 – 9,999.99 range
struct_zone = flows_enriched[flows_enriched['amount'].between(9000, 9999.99)]
print('Suspicious flows (structuring):', len(struct_zone))
struct_zone[['flow_id', 'client_id', 'date', 'operation_type', 'amount']].head()

In [ ]:
# 2. Most frequent operation type
struct_zone['operation_type'].value_counts()

In [ ]:
# 3. Clients with >= 2 flows in the 9,000–9,999 range
struct_counts = struct_zone.groupby('client_id').agg(
    nb_flows=('flow_id', 'count'),
    total_amount=('amount', 'sum'),
    risk_level=('kyc_risk_level', 'first'),
).reset_index()
struct_counts[struct_counts['nb_flows'] >= 2].sort_values('nb_flows', ascending=False)

In [ ]:
# 4. structuring_alert column
flows_enriched['structuring_alert'] = (
    flows_enriched['amount'].between(9000, 9999.99)
    & flows_enriched['operation_type'].str.startswith('Cash')
)
print('Structuring alerts:', flows_enriched['structuring_alert'].sum())

---
## Exercise 4 — Cash flow analysis

In [ ]:
# 1. Cash flows
cash_flows = flows_enriched[
    flows_enriched['operation_type'].isin(['Cash Deposit', 'Cash Withdrawal'])
]
print('Cash flows:', len(cash_flows))

In [ ]:
# 2. Top 10 clients by cash total
top_cash = (
    cash_flows.groupby('client_id')
    .agg(cash_total=('amount', 'sum'), kyc=('kyc_risk_level', 'first'))
    .sort_values('cash_total', ascending=False)
    .head(10)
    .round(2)
)
top_cash

In [ ]:
# 3. cash_alert column
flows_enriched['cash_alert'] = (
    flows_enriched['operation_type'].isin(['Cash Deposit', 'Cash Withdrawal'])
    & (flows_enriched['amount'] > 10000)
)
print('Cash alerts:', flows_enriched['cash_alert'].sum())

In [ ]:
# 4. Number of distinct clients with a cash alert
clients_cash_alerts = flows_enriched[flows_enriched['cash_alert']]['client_id'].nunique()
print('Clients with a cash alert:', clients_cash_alerts)

---
## Exercise 5 — Flows to risk countries

In [ ]:
# 1. risk_country column
risk_country_list = ['AE', 'PA', 'CY']
flows_enriched['risk_country'] = flows_enriched['counterparty_country'].isin(risk_country_list)
print('Flows to risk countries:', flows_enriched['risk_country'].sum())

In [ ]:
# 2. Total amount to risk countries by operation type
(
    flows_enriched[flows_enriched['risk_country']]
    .groupby('operation_type')['amount']
    .sum()
    .round(2)
    .sort_values(ascending=False)
)

In [ ]:
# 3. Clients with >= 3 flows to risk countries
risk_flows = flows_enriched[flows_enriched['risk_country']]
risk_recap = risk_flows.groupby('client_id').agg(
    risk_flow_count=('flow_id', 'count'),
    risk_total_amount=('amount', 'sum'),
    segment=('segment', 'first'),
    kyc=('kyc_risk_level', 'first'),
).reset_index()
risk_recap[risk_recap['risk_flow_count'] >= 3].sort_values('risk_flow_count', ascending=False)

In [ ]:
# 4. High KYC clients in flows to risk countries
flows_enriched[
    flows_enriched['risk_country']
    & (flows_enriched['kyc_risk_level'] == 'High')
][['client_id', 'date', 'operation_type', 'amount', 'counterparty_country', 'segment']].drop_duplicates()

---
## Exercise 6 — Multi-criteria risk scoring

In [ ]:
# Aggregates per client
scoring = flows_enriched.groupby('client_id').agg(
    total_flows=('flow_id', 'count'),
    total_amount=('amount', 'sum'),
    structuring_alerts_count=('structuring_alert', 'sum'),
    cash_alerts_count=('cash_alert', 'sum'),
    risk_country_flows=('risk_country', 'sum'),
    kyc_risk_level=('kyc_risk_level', 'first'),
    segment=('segment', 'first'),
).reset_index()

scoring['risk_score'] = (
    scoring['structuring_alerts_count']
    + scoring['cash_alerts_count']
    + scoring['risk_country_flows']
)
scoring['total_amount'] = scoring['total_amount'].round(2)

top10 = scoring.sort_values('risk_score', ascending=False).head(10)
top10

In [ ]:
# Excel export
top10.to_excel('aml_risk_report.xlsx', index=False)
print('File aml_risk_report.xlsx created with', len(top10), 'rows.')